<h2><b><center><span style="color:blue">OPENAI RESPONSES API

A beginner-friendly handbook | From basic LLM calls to production AI agents

Think of the Responses API as a way to communicate with an AI model, give it tools, maintain context, and build intelligent workflows, all through a unified interface.

We'll follow one practical example throughout: extracting commercial information from an email.

### <span style="color: magenta;">1. First, understand the big picture</span>

Imagine receiving this email:

Vendor funding confirmation

The vendor has agreed to provide $50,000 for Q3. Please proceed with creating the DNT.

An AI application might need to:

1. Understand the email.
2. Extract the amount, period, and commercial status.
3. Look up the corresponding deal, if required.
4. Return a structured JSON response.

The Responses API helps your application interact with the model to perform these operations.

The API provides access to model capabilities; it is not an LLM itself.

### <span style="color: magenta;">2. Why Responses API when Chat Completions already exists?</span>

Chat Completions is already capable of generating text, extracting entities, returning structured outputs, and calling functions.

The problem is that sophisticated agents need more than a simple question-answer interaction.

<span style="color: limegreen;">Chat Completions: Message-oriented</span>

You send conversation messages and receive model responses. Your application usually manages conversation history and tool execution.

<span style="color: limegreen;">Responses API: Response and tool-oriented</span>

It additionally supports integrated tools, linked response history, and more complete multi-step reasoning workflows.

<span style="color: lightblue;">What actually stands out?</span>

| <span style="color: lightblue;">Capability</span> | <span style="color: lightblue;">Chat Completions</span> | <span style="color: lightblue;">Responses</span> |
| ---------- | ---------------- | --------- |
| <span style="color: limegreen;">Text generation</span> | Yes | Yes |
| <span style="color: limegreen;">Structured output</span> | Yes | Yes |
| <span style="color: limegreen;">Function calling</span> | Yes | Yes |
| <span style="color: limegreen;">Built-in tool ecosystem</span> | Limited | Broader |
| <span style="color: limegreen;">Previous response continuation</span> | Resend history | `previous_response_id` |
| <span style="color: limegreen;">Reasoning state continuity</span> | More manual | Better support |
| <span style="color: limegreen;">Integrated tool execution</span> | Limited | Yes, for supported tools |
| <span style="color: limegreen;">Agentic applications</span> | Possible | More convenient |

Remember: Responses is not inherently more accurate than Chat Completions. It primarily simplifies building sophisticated AI applications.

### <span style="color: magenta;">3. Your first Responses API call</span>

```python
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="YOUR_MODEL",
    instructions="Extract commercial information.",
    input="Vendor confirmed $50,000 for Q3."
)
print(response.output_text)
```

Three things to understand:

- `model`: Which model should process the request?
- `instructions`: How should the model behave?
- `input`: What content should the model process?

`response.output_text` conveniently retrieves generated text. The full `response.output` can also contain tool calls and other items.

<span style="color: lightblue;">What changes in Chat Completions?</span>

| <span style="color: lightblue;">API</span> | <span style="color: lightblue;">Input format</span> | <span style="color: lightblue;">Read output</span> |
| --- | ------------ | ----------- |
| <span style="color: limegreen;">Chat Completions</span> | `messages=[...]` | `choices[0].message.content` |
| <span style="color: limegreen;">Responses</span> | `input=...` | `output_text` |

Responses also accepts role-based message arrays in `input`. So it's not limited to plain strings.

### <span style="color: magenta;">4. The features that make Responses API interesting</span>

<span style="color: limegreen;">A. Reasoning capabilities</span>

Reasoning models can spend additional computation analysing complex problems before answering.

`reasoning={"effort": "medium"}`

Useful for interpreting ambiguous commercial agreements, multi-step decisions, and contextual analysis. Greater reasoning effort may increase latency and cost.

<span style="color: limegreen;">B. Conversation continuity</span>

With `previous_response_id`, you can link the next request to an earlier stored response without rebuilding the entire history yourself.

The context is still subject to token limits and billing. Previous instructions should be supplied again when needed.

<span style="color: limegreen;">C. Built-in tools</span>

Supported tools include web search, file search, code interpreter, shell, and MCP integrations.

Some tools execute within the API workflow, reducing the amount of orchestration code needed.

<span style="color: limegreen;">D. Structured outputs</span>

Define a JSON Schema or Pydantic model so the output follows a predictable structure.

Particularly useful for pipelines writing to Delta tables or databases. Schema compliance does not guarantee factual accuracy.

<span style="color: limegreen;">E. Prompt caching</span>

Reuses eligible repeated prompt prefixes to reduce input processing costs and latency.

Most useful when many requests share the same large system prompt or domain instructions.

### <span style="color: magenta;">5. Understand how tool calling works</span>

There are two different mechanisms that beginners often confuse.

<span style="color: limegreen;">Built-in tool execution</span>

For supported built-in tools, the API manages the internal execution sequence.

<span style="color: limegreen;">Custom function calling</span>

Model requests function → Your Python application executes it → You return the result → Model continues.

<span style="color: lightblue;">Example</span>: To retrieve a deal from an internal database, define a custom function such as `get_deal_details(deal_id)`. The model can request it, but your application must execute the actual database query.

Tool calling does not automatically make every application an autonomous agent.

### <span style="color: magenta;">6. What about Pydantic and `beta` in Chat Completions?</span>

Both APIs support structured parsing.

<span style="color: limegreen;">Responses API</span>

```python
response = client.responses.parse(
    model="YOUR_MODEL",
    input=email_content,
    text_format=AmountRecord
)
data = response.output_parsed
```

<span style="color: limegreen;">Chat Completions API</span>

```python
response = client.beta.chat.completions.parse(
    model="YOUR_MODEL",
    messages=[
        {"role": "user", "content": email_content}
    ],
    response_format=AmountRecord
)
data = response.choices[0].message.parsed
```

Does Chat Completions have beta? Yes. `client.beta.chat.completions.parse()` is a Python SDK parsing helper under the `beta` namespace. It does not mean that the entire Chat Completions API is a beta product.

Also, `.parse()` is a convenient SDK method. The underlying APIs support structured output configuration independently.

### <span style="color: magenta;">7. Production parameters worth remembering</span>

Instead of memorising every API parameter, group them by purpose.

| <span style="color: lightblue;">Purpose</span> | <span style="color: lightblue;">Important parameters</span> |
| ------- | -------------------- |
| <span style="color: limegreen;">Model and input</span> | `model`, `input`, `instructions` |
| <span style="color: limegreen;">Control generation</span> | `max_output_tokens`, `reasoning` |
| <span style="color: limegreen;">Tools</span> | `tools`, `tool_choice` |
| <span style="color: limegreen;">Structured data</span> | `text.format`, SDK `text_format` |
| <span style="color: limegreen;">Conversation continuity</span> | `previous_response_id`, `conversation` |
| <span style="color: limegreen;">Performance</span> | `stream`, `background` |
| <span style="color: limegreen;">Cost optimisation</span> | `prompt_cache_key`, `prompt_cache_retention` |
| <span style="color: limegreen;">Storage</span> | `store` |
| <span style="color: limegreen;">Monitoring</span> | Response `usage`, `status`, `id` |

Availability varies with model, SDK, and platform.

<span style="color: lightblue;">Three performance features, explained simply</span>

- Streaming: Receive generated output incrementally rather than all at once.
- Background mode: Let eligible long-running requests execute asynchronously and retrieve their status later.
- Prompt caching: Reuse matching prompt prefixes. This is not response caching; the model still generates a new answer.

### <span style="color: magenta;">8. How does Responses API fit with LangGraph?</span>

LangGraph determines the workflow. Responses API handles model interactions inside that workflow.

LangGraph gives you explicit control over the sequence, branching, state, retries, and recovery. Responses API provides model capabilities.

A workflow containing several extraction functions is not automatically agentic. Autonomy depends on whether the model can make meaningful decisions about actions, tools, and next steps.

### <span style="color: magenta;">9. Where Responses API is useful</span>

<span style="color: limegreen;">Information extraction</span>

Extract amounts, periods, statuses, and deal IDs from unstructured emails into structured records.

<span style="color: limegreen;">Document-based assistants</span>

Search documents and answer questions using retrieved evidence.

<span style="color: limegreen;">Agentic automation</span>

Let a model select supported tools and perform multi-step tasks within controlled permissions.

<span style="color: limegreen;">Reconciliation and validation</span>

Interpret evidence, query business systems through functions, and produce candidate matches for rule-based verification.

### <span style="color: magenta;">10. Where it falls short</span>

- Token costs: Reasoning, tool outputs, and long conversation histories can become expensive.
- Latency: Additional reasoning and tool calls increase processing time.
- Hallucinations: Structured output can still contain incorrect or unsupported values.
- State limitations: Conversation continuity depends on supported storage and retention.
- Operational complexity: Tool permissions, retries, incomplete responses, and errors still need handling.
- Platform differences: OpenAI and Azure deployments may support different tool combinations and features.

For production, measure extraction precision and recall, validate against source evidence, log token usage and failures, and use deterministic rules wherever possible.

### <span style="color: magenta;">11. Test your understanding</span>

<span style="color: limegreen;">Quick knowledge check</span>

0 of 4 answered

1. Which API is generally more convenient for built-in tool-heavy workflows?

   - Chat Completions
   - Responses API
   - Neither supports tools

2. Does Pydantic guarantee that an extracted amount is correct?

   - Yes
   - No, only schema constraints are enforced
   - Only with high reasoning

3. Does previous_response_id make earlier input tokens free?

   - Yes
   - No

4. Who controls graph-level retries and state in a LangGraph application?

   - LangGraph
   - Pydantic
   - Prompt caching

### <span style="color: magenta;">12. The five things to retain</span>

1. Responses API is the unified model interface, especially useful for reasoning and tool-enabled applications.
2. Chat Completions remains relevant and supports structured outputs, reasoning models, and function calling.
3. `previous_response_id` manages conversation continuity, not free context or permanent memory.
4. Pydantic controls output structure, while prompts, source evidence, and validation determine extraction reliability.
5. Responses API + LangGraph is a useful production combination, but neither automatically guarantees a reliable agent.

For further reading: [Responses migration guide](https://developers.openai.com/api/docs/guides/migrate-to-responses) and [Responses API reference](https://developers.openai.com/api/reference/resources/responses).

The key decision: Choose Responses for the capabilities your application genuinely needs, not simply because it is newer.